In [1]:
# Install DeepMReye
!pip install deepmreye

In [13]:
import os
import nibabel as nib
import numpy as np

from deepmreye import preprocess, train, analyse
from deepmreye.util import model_opts, data_generator
from deepmreye.preprocess import get_masks, run_participant, save_data, normalize_img
from deepmreye.architecture import create_standard_model
from deepmreye.analyse import visualise_predictions_slider
from sklearn.model_selection import train_test_split
from nibabel import load
#from funcs import load_label, process_one_subject_session, process_all_subjects, save_test_data
from funcs import save_test_data

In [14]:
# Define paths
base_path = "/data2/2104/derivatives/"
calibration_path = os.path.join(base_path, "deepmreye/calibration_imgs")
movie_path = os.path.join(base_path, "fmriprep")
calibration_npz = os.path.join(base_path, "deepmreye/calibration_npz")
movie_npz = os.path.join(base_path, "deepmreye/movie_npz")
model_weights = os.path.join(base_path, "deepmreye/model_weights")
calibration_labels = os.path.join(base_path, "deepmreye/stim_vals_sub-TR.csv")

In [9]:
def process_all_subjects_movie():
    
    # Get list of all subjects
    subjects = sorted([d for d in os.listdir(movie_path) 
                      if os.path.isdir(os.path.join(movie_path, d)) and d.startswith('sub-')])

    # Load masks once
    eyemask_small, eyemask_big, dme_template, mask, x_edges, y_edges, z_edges = get_masks()

    # Dictionary to track processing results
    processing_results = {
        'success': [],
        'failed': [],
        'skipped': [],
        'already_exists': [] 
    }

    for subject in subjects:
        subject_path = os.path.join(movie_path, subject)
        sessions = [d for d in os.listdir(subject_path) 
                   if os.path.isdir(os.path.join(subject_path, d)) and d.startswith('ses-')]
        
        for session in sessions:
            session_path = os.path.join(subject_path, session, 'func')
            
            if not os.path.exists(session_path):
                print(f"Session func directory missing: {session_path}")
                processing_results['skipped'].append(f"{subject}_{session}")
                continue
            
            # Find all movie run files for this session (runs 1-3)
            for run in ['1', '2', '3']:
                nii_filename = f"{subject}_{session}_task-movie_run-{run}_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz"
                nii_path = os.path.join(session_path, nii_filename)
                
                if not os.path.exists(nii_path):
                    print(f"NIfTI file missing: {nii_path}")
                    processing_results['skipped'].append(f"{subject}_{session}_run-{run}")
                    continue
                
                data_key = f"{subject}_{session}_run-{run}"
                
                # Check if output npz file already exists
                npz_filename = f"{data_key}.npz"
                npz_path = os.path.join(movie_npz, npz_filename)
                
                if os.path.exists(npz_path):
                    print(f"Output file already exists, skipping: {npz_path}")
                    processing_results['already_exists'].append(data_key)
                    continue
                
                try:
                
                    # Process NIfTI
                    masked_eye_data, transform_stats = run_participant(
                        nii_path, dme_template, eyemask_big,
                        eyemask_small, x_edges, y_edges, z_edges
                    )
                    masked_eye_data = normalize_img(masked_eye_data)
                    
                    # Save - each run gets its own npz file
                    T = masked_eye_data.shape[3]  
                    ids_this_run = np.repeat(data_key, T)[None, :]
                                        
                    save_test_data(
                        participant=f"{data_key}.npz",
                        participant_data=[masked_eye_data],
                        participant_ids=[ids_this_run],
                        processed_data=movie_npz
                    )
                
                    processing_results['success'].append(data_key)
                    print(f"Successfully processed {data_key}")
                
                except Exception as e:
                    print(f"Error processing {data_key}: {e}")
                    processing_results['failed'].append(data_key)

    # Print summary
    print("\nProcessing Summary:")
    print(f"Successfully processed: {len(processing_results['success'])}")
    print(f"Failed: {len(processing_results['failed'])}")
    print(f"Skipped: {len(processing_results['skipped'])}")
    print(f"Already exists: {len(processing_results['already_exists'])}")
    
    return processing_results

process_all_subjects_movie()

Output file already exists, skipping: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-3_run-1.npz
Output file already exists, skipping: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-3_run-2.npz
Output file already exists, skipping: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-3_run-3.npz
Output file already exists, skipping: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-4_run-1.npz
Output file already exists, skipping: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-4_run-2.npz
Output file already exists, skipping: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040005_ses-4_run-3.npz
Output file already exists, skipping: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040010_ses-3_run-1.npz
Output file already exists, skipping: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040010_ses-3_run-2.npz
Output file already exists, skipping: /data2/2104/derivatives/deepmreye/movie_npz/sub-21040010_ses-3_run-3.npz
O

KeyboardInterrupt: 

In [15]:
def load_label(label_path, label_type="calibration_run", calibration=True):
    """
    Load labels and return shape (T, 10, 2): 10 samples per TR, (x, y) per sample.
    Handles both cases:
      - CSV with 2 columns (x,y) per TR → repeats to (10,2)
      - CSV with 20 columns (x1,y1,...,x10,y10) per TR → reshapes to (10,2)
    """
    labels = np.genfromtxt(label_path, delimiter=",")
    labels = labels[1:]  # drop header row if present
    labels = labels.astype(float)

    if labels.ndim != 2:
        raise ValueError(f"Expected 2D labels array, got {labels.ndim}D")

    T, C = labels.shape
    if C == 2:
        # old format: one (x,y) per TR → repeat to 10 samples
        this_label = labels[:, None, :]             # (T,1,2)
        this_label = np.repeat(this_label, 10, 1)   # (T,10,2)
    elif C == 20:
        # new format: (x1,y1,...,x10,y10) per TR → reshape to (10,2)
        this_label = labels.reshape(T, 10, 2)       # (T,10,2)
    else:
        raise ValueError(f"Unexpected number of columns in labels: {C} (expected 2 or 20)")

    # Normalize to [-0.5, 0.5] assuming original range [-0.95, 0.95]
    this_label = (this_label - (-0.95)) / (0.95 - (-0.95))
    this_label -= 0.5

    # Flip Y for this dataset
    this_label[..., 1] *= -1

    # Convert to visual angles
    this_label[..., 0] *= 19.0
    this_label[..., 1] *= 14.7

    return this_label

In [16]:
# run the processing for one subject
# processing_results = process_one_subject_session("sub-21040005", "ses-3", True)

# run the processing for all subjects
#processing_results = process_all_subjects(True)

In [17]:
def process_all_subjects(calibration):
    # Get list of all subjects
    subjects = sorted([d for d in os.listdir(calibration_path) 
                      if os.path.isdir(os.path.join(calibration_path, d)) and d.startswith('sub-')])

    # Load masks once
    eyemask_small, eyemask_big, dme_template, mask, x_edges, y_edges, z_edges = get_masks()

    # Dictionary to track processing results
    processing_results = {
        'success': [],
        'failed': [],
        'skipped': []
    }

    for subject in subjects:
        subject_path = os.path.join(calibration_path, subject)
        sessions = [d for d in os.listdir(subject_path) 
                   if os.path.isdir(os.path.join(subject_path, d)) and d.startswith('ses-')]
        
        for session in sessions:
            data_key = f"{subject}_{session}"
            nii_path = os.path.join(subject_path, session, f"{data_key}_calibration.nii")
            
            if not os.path.exists(nii_path):
                print(f"NIfTI file missing: {nii_path}")
                processing_results['skipped'].append(data_key)
                continue

            try:
                # 1) Load labels
                if calibration == True:
                    labels = load_label(label_path=calibration_labels, label_type="calibration_run", calibration=True)
                else:
                    labels = load_label(label_path=movie_labels, label_type="movie_run", calibration=False)
            
                # 2) Process NIfTI
                masked_eye_data, transform_stats = run_participant(
                    nii_path, dme_template, eyemask_big,
                    eyemask_small, x_edges, y_edges, z_edges
                )
                masked_eye_data = normalize_img(masked_eye_data)
            
                # 3) Sanity checks
                if masked_eye_data.ndim != 4:
                    raise ValueError(f"Expected 4D data, got {masked_eye_data.ndim}D")
                if labels.shape[0] != masked_eye_data.shape[3]:
                    raise ValueError(
                        f"Label mismatch ({labels.shape[0]} vs {masked_eye_data.shape[3]} volumes)"
                    )
            
                # 4) Save
                T = labels.shape[0]
                ids_this_run = np.repeat(data_key, T)[None, :]
                
                os.makedirs(calibration_npz, exist_ok=True)

                if calibration == True:
                    save_data(
                        participant=f"{data_key}.npz",
                        participant_data=[masked_eye_data],
                        participant_labels=[labels],
                        participant_ids=[ids_this_run],
                        processed_data=calibration_npz
                    )
                else:
                    save_data(
                        participant=f"{data_key}.npz",
                        participant_data=[masked_eye_data],
                        participant_labels=[labels],
                        participant_ids=[ids_this_run],
                        processed_data=movie_npz
                    )
            
                processing_results['success'].append(data_key)
                print(f"Successfully processed {data_key}")
            
            except Exception as e:
                print(f"Error processing {data_key}: {e}")
                processing_results['failed'].append(data_key)


    # Print summary
    print("\nProcessing Summary:")
    print(f"Successfully processed: {len(processing_results['success'])}")
    print(f"Failed: {len(processing_results['failed'])}")
    print(f"Skipped: {len(processing_results['skipped'])}")
    
    return processing_results

In [18]:
processing_results = process_all_subjects(True)

Mask 0/2, Sum: 3.856, Mean 0.321, Std 0.440, Median 0.005
Mask 1/2, Sum: 9.787, Mean 0.816, Std 3.309, Median 0.025
Mask 2/2, Sum: -0.694, Mean -0.058, Std 1.112, Median 0.005
Saving eye data (75, 47, 29, 18) and targets (75, 10, 2) (NaN 0) to file /data2/2104/derivatives/deepmreye/calibration_npz/sub-21040005_ses-3.npz
Successfully processed sub-21040005_ses-3
NIfTI file missing: /data2/2104/derivatives/deepmreye/calibration_imgs/sub-21040005/ses-4/sub-21040005_ses-4_calibration.nii
Mask 0/2, Sum: 1.973, Mean 0.164, Std 0.537, Median 0.005
Mask 1/2, Sum: 4.211, Mean 0.351, Std 2.380, Median 0.026
Mask 2/2, Sum: -4.052, Mean -0.338, Std 1.225, Median -0.000
Saving eye data (75, 47, 29, 18) and targets (75, 10, 2) (NaN 0) to file /data2/2104/derivatives/deepmreye/calibration_npz/sub-21040016_ses-3.npz
Successfully processed sub-21040016_ses-3
Mask 0/2, Sum: 3.323, Mean 0.277, Std 0.432, Median 0.006
Mask 1/2, Sum: 7.711, Mean 0.643, Std 1.530, Median 0.034
Mask 2/2, Sum: -4.355, Mean -0